# Notebook 04 — Building a Reproducible Benchmark Harness

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/04_building_a_reproducible_benchmark_harness.ipynb)

This notebook turns the earlier mental models into a repeatable benchmark harness. The focus is not on any one runtime. It is on measurement discipline: fixed datasets, deterministic settings, warmup control, clear metrics, and reusable artifacts.

## What you will learn

- how to structure benchmark cases and runtime configs
- how to separate warmup from steady-state timing
- how to capture latency, throughput, and memory in one report
- how to save artifacts that later notebooks can compare

## Reproducibility principle

A benchmark without a stable harness is just a story. The harness should make it hard to accidentally change seeds, workloads, or timing rules without noticing.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
import os
import hashlib
from time import perf_counter, sleep

random.seed(4)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "module_04_benchmark_harness"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
PROCESS = psutil.Process(os.getpid())

def percentile(values, pct):
    if len(values) == 0:
        return 0.0
    return float(np.percentile(values, pct))

def benchmark_id(case_id, runtime_id, run_index):
    payload = f"{case_id}|{runtime_id}|{run_index}".encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:12]

def current_rss_mb():
    return round(PROCESS.memory_info().rss / (1024 ** 2), 2)

print("Artifact root:", NOTEBOOK_ROOT.resolve())

## Step 1: Define benchmark cases

A good case list is small enough to understand and broad enough to expose important runtime differences. We will cover short chat, retrieval-heavy prompts, long outputs, and structured extraction.

In [ ]:
benchmark_cases = pd.DataFrame([
    {"case_id": "chat_short", "family": "chat", "prompt_tokens": 512, "output_tokens": 96, "shared_prefix_ratio": 0.20},
    {"case_id": "rag_medium", "family": "retrieval", "prompt_tokens": 2400, "output_tokens": 128, "shared_prefix_ratio": 0.55},
    {"case_id": "code_edit", "family": "coding", "prompt_tokens": 1800, "output_tokens": 240, "shared_prefix_ratio": 0.35},
    {"case_id": "json_extract", "family": "structured", "prompt_tokens": 1400, "output_tokens": 80, "shared_prefix_ratio": 0.75},
])
benchmark_cases

## Step 2: Define runtime profiles

These profiles are synthetic, but each one stands for a recognizable serving shape: local baseline, throughput-optimized server, and schema-focused server.

In [ ]:
runtime_profiles = pd.DataFrame([
    {"runtime_id": "baseline_python", "prefill_coeff": 0.033, "decode_ms": 2.2, "batch_gain": 1.00, "cache_gain": 0.00, "schema_bonus": 0.00},
    {"runtime_id": "throughput_server", "prefill_coeff": 0.026, "decode_ms": 1.65, "batch_gain": 0.72, "cache_gain": 0.25, "schema_bonus": 0.00},
    {"runtime_id": "structured_server", "prefill_coeff": 0.027, "decode_ms": 1.75, "batch_gain": 0.78, "cache_gain": 0.30, "schema_bonus": 0.08},
])
runtime_profiles

## Step 3: Build a tiny timed workload

We use a tiny `sleep`-based workload so the harness measures real wall-clock time while staying lightweight. The synthetic latency model remains deterministic because the sleep duration is derived from the case and runtime profile.

In [ ]:
def synthetic_run(case_row, runtime_row, warmed=False, run_index=0):
    seed = "{}|{}|{}|{}".format(case_row["case_id"], runtime_row["runtime_id"], warmed, run_index)
    rng = random.Random(seed)
    effective_prompt = case_row["prompt_tokens"] * (1.0 - case_row["shared_prefix_ratio"] * runtime_row["cache_gain"])
    prefill_ms = 16.0 + runtime_row["prefill_coeff"] * (effective_prompt ** 1.1)
    decode_ms = case_row["output_tokens"] * runtime_row["decode_ms"]
    warmup_penalty_ms = 45.0 if not warmed else 0.0
    jitter_ms = rng.uniform(-6.0, 6.0)
    total_ms = max(6.0, prefill_ms + decode_ms + warmup_penalty_ms + jitter_ms)
    memory_probe = np.ones((256, 64), dtype=np.float32) * (1 + case_row["prompt_tokens"] / 4096.0)
    rss_before = current_rss_mb()
    started = perf_counter()
    sleep(total_ms / 1000.0 / 40.0)
    elapsed_ms = (perf_counter() - started) * 1000.0 * 40.0
    rss_after = current_rss_mb()
    del memory_probe
    return {
        "prefill_ms_est": round(prefill_ms, 2),
        "decode_ms_est": round(decode_ms, 2),
        "elapsed_ms": round(elapsed_ms, 2),
        "peak_rss_mb": max(rss_before, rss_after),
        "warmup_penalty_ms": warmup_penalty_ms,
    }

In [ ]:
example = synthetic_run(benchmark_cases.iloc[0].to_dict(), runtime_profiles.iloc[0].to_dict(), warmed=False, run_index=0)
example

## Step 4: Separate warmup from steady state

Warmup is real, but it should not be mixed silently with steady-state reporting. Good benchmark harnesses either report it separately or discard it deliberately.

In [ ]:
def run_benchmark(case_row, runtime_row, repeats=6, warmup_runs=1):
    rows = []
    for run_index in range(warmup_runs + repeats):
        warmed = run_index >= warmup_runs
        result = synthetic_run(case_row, runtime_row, warmed=warmed, run_index=run_index)
        rows.append({
            "run_id": benchmark_id(case_row["case_id"], runtime_row["runtime_id"], run_index),
            "case_id": case_row["case_id"],
            "runtime_id": runtime_row["runtime_id"],
            "phase": "steady" if warmed else "warmup",
            **result,
        })
    return pd.DataFrame(rows)

In [ ]:
benchmark_runs = []
for case_row in benchmark_cases.to_dict("records"):
    for runtime_row in runtime_profiles.to_dict("records"):
        benchmark_runs.append(run_benchmark(case_row, runtime_row, repeats=5, warmup_runs=1))

benchmark_runs = pd.concat(benchmark_runs, ignore_index=True)
benchmark_runs.head(12)

## Step 5: Summarize latency correctly

Use percentiles and keep warmup explicit. Means alone are not enough, especially once load or batching enters the picture.

In [ ]:
steady = benchmark_runs[benchmark_runs["phase"] == "steady"].copy()
latency_summary = steady.groupby(["case_id", "runtime_id"]).agg(
    mean_elapsed_ms=("elapsed_ms", "mean"),
    p50_elapsed_ms=("elapsed_ms", lambda values: percentile(values, 50)),
    p95_elapsed_ms=("elapsed_ms", lambda values: percentile(values, 95)),
    mean_peak_rss_mb=("peak_rss_mb", "mean"),
).round(2).reset_index()
latency_summary

## Step 6: Add a simple throughput harness

A throughput harness answers a different question: how much work can the runtime finish when multiple requests are in flight? We can approximate that without a full server by combining per-request estimates with a batch-gain factor.

In [ ]:
def estimate_throughput(case_row, runtime_row, concurrent_requests):
    base = synthetic_run(case_row, runtime_row, warmed=True, run_index=999)
    effective_elapsed_ms = base["elapsed_ms"] * runtime_row["batch_gain"] * (1 + 0.04 * max(0, concurrent_requests - 1))
    requests_per_second = concurrent_requests / max(effective_elapsed_ms / 1000.0, 1e-6)
    tokens_per_second = requests_per_second * case_row["output_tokens"]
    return {
        "concurrent_requests": concurrent_requests,
        "effective_elapsed_ms": round(effective_elapsed_ms, 2),
        "requests_per_second": round(requests_per_second, 2),
        "tokens_per_second": round(tokens_per_second, 2),
    }

In [ ]:
throughput_rows = []
for case_row in benchmark_cases.to_dict("records"):
    for runtime_row in runtime_profiles.to_dict("records"):
        for concurrent_requests in [1, 2, 4, 8]:
            throughput_rows.append({
                "case_id": case_row["case_id"],
                "runtime_id": runtime_row["runtime_id"],
                **estimate_throughput(case_row, runtime_row, concurrent_requests),
            })

throughput_df = pd.DataFrame(throughput_rows)
throughput_df.head(12)

## Step 7: Visualize throughput versus latency

The interesting comparison is often the frontier: which runtime gives more throughput for similar latency, or lower latency for similar throughput?

In [ ]:
frontier = throughput_df[throughput_df["concurrent_requests"] == 4].merge(
    latency_summary[["case_id", "runtime_id", "p95_elapsed_ms"]],
    on=["case_id", "runtime_id"],
    how="left",
)
fig, ax = plt.subplots(figsize=(9, 5))
for runtime_id, frame in frontier.groupby("runtime_id"):
    ax.scatter(frame["p95_elapsed_ms"], frame["tokens_per_second"], s=90, label=runtime_id)
    for _, row in frame.iterrows():
        ax.text(row["p95_elapsed_ms"] + 3, row["tokens_per_second"] + 2, row["case_id"], fontsize=8)
ax.set_xlabel("p95 steady-state latency ms")
ax.set_ylabel("tokens per second at concurrency=4")
ax.set_title("Benchmark frontier")
ax.legend()
plt.show()

## Step 8: Make determinism testable

Synthetic or real, a benchmark harness should make deterministic mode explicit. If repeated runs under the same seed drift wildly, either the system or the harness is unstable.

In [ ]:
repeat_a = run_benchmark(benchmark_cases.iloc[1].to_dict(), runtime_profiles.iloc[1].to_dict(), repeats=3, warmup_runs=0)
repeat_b = run_benchmark(benchmark_cases.iloc[1].to_dict(), runtime_profiles.iloc[1].to_dict(), repeats=3, warmup_runs=0)
determinism_check = repeat_a[["run_id", "elapsed_ms"]].merge(repeat_b[["run_id", "elapsed_ms"]], on="run_id", suffixes=("_a", "_b"))
determinism_check["exact_match"] = determinism_check["elapsed_ms_a"] == determinism_check["elapsed_ms_b"]
determinism_check

## Step 9: Add regression gates

Benchmarks become operationally useful once they can say pass or fail. A gate turns “interesting chart” into “safe to ship” or “not yet.”

In [ ]:
baseline = latency_summary[latency_summary["runtime_id"] == "baseline_python"][["case_id", "p95_elapsed_ms"]].rename(columns={"p95_elapsed_ms": "baseline_p95_ms"})
challenger = latency_summary[latency_summary["runtime_id"] == "throughput_server"][["case_id", "p95_elapsed_ms"]].rename(columns={"p95_elapsed_ms": "challenger_p95_ms"})
gate_df = baseline.merge(challenger, on="case_id")
gate_df["delta_pct"] = ((gate_df["challenger_p95_ms"] - gate_df["baseline_p95_ms"]) / gate_df["baseline_p95_ms"] * 100).round(2)
gate_df["gate_pass"] = gate_df["delta_pct"] <= 5.0
gate_df

## Step 10: Save raw and summary artifacts

A benchmark run should leave behind raw rows, summaries, and enough metadata to reproduce the run later.

In [ ]:
benchmark_runs.to_csv(NOTEBOOK_ROOT / "benchmark_runs.csv", index=False)
latency_summary.to_csv(NOTEBOOK_ROOT / "latency_summary.csv", index=False)
throughput_df.to_csv(NOTEBOOK_ROOT / "throughput_summary.csv", index=False)
gate_df.to_csv(NOTEBOOK_ROOT / "regression_gate.csv", index=False)

artifact_manifest = {
    "cases": benchmark_cases.to_dict("records"),
    "runtimes": runtime_profiles.to_dict("records"),
    "created_files": [
        "benchmark_runs.csv",
        "latency_summary.csv",
        "throughput_summary.csv",
        "regression_gate.csv",
    ],
}
with (NOTEBOOK_ROOT / "artifact_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(artifact_manifest, handle, indent=2)

print("Saved artifacts to", NOTEBOOK_ROOT.resolve())

## Step 11: Create a compact report card

A small report card is often more useful than a long notebook when sharing results with a team. We will export a compact markdown-style summary as plain text.

In [ ]:
report_lines = [
    "Benchmark report",
    "================",
    "",
    "Latency summary:",
]
for row in latency_summary.to_dict("records"):
    report_lines.append(
        "- {case_id} / {runtime_id}: mean={mean_elapsed_ms} ms, p95={p95_elapsed_ms} ms, rss={mean_peak_rss_mb} MB".format(**row)
    )
report_lines.append("")
report_lines.append("Regression gates:")
for row in gate_df.to_dict("records"):
    report_lines.append(
        "- {case_id}: delta={delta_pct}% pass={gate_pass}".format(**row)
    )

report_path = NOTEBOOK_ROOT / "benchmark_report.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(report_path.read_text(encoding="utf-8"))

## Step 12: Leave a hook for real runtimes

Later notebooks can replace `synthetic_run` with a real client call while keeping the same harness shape. That separation is the whole point of the design.

In [ ]:
class RuntimeAdapter:
    def __init__(self, runtime_id):
        self.runtime_id = runtime_id

    def generate(self, prompt, max_new_tokens=64):
        raise NotImplementedError("Replace with a real runtime call in later notebooks")

adapter = RuntimeAdapter("example-runtime")
print("Adapter placeholder ready for later modules:", adapter.runtime_id)

## Exercises

1. Add a new benchmark case with a much longer prompt and see which runtime profile benefits most from caching.
2. Tighten the regression gate from 5% to 0% and observe which cases fail.
3. Replace the synthetic memory probe with your own measured allocation if you have a local runtime available.

In [ ]:
exercise_template = pd.DataFrame([
    {"field": "new_case_id", "value": ""},
    {"field": "target_runtime", "value": ""},
    {"field": "pass_threshold_pct", "value": ""},
])
exercise_template

## Recap

A reproducible benchmark harness has stable cases, explicit runtime configs, deterministic execution rules, warmup handling, latency summaries, throughput summaries, memory capture, and saved artifacts. Later runtime notebooks will plug real engines into this exact skeleton.

In [ ]:
assert determinism_check["exact_match"].all()
assert gate_df["gate_pass"].isin([True, False]).all()
assert (NOTEBOOK_ROOT / "artifact_manifest.json").exists()
print("Notebook 04 sanity checks passed.")